# Stuff+ Pitch Quality Model
## SABR Analytics Level IV Capstone — Jordan Kurzweil

---

### Project Overview

**Stuff+** is a pitch quality metric that evaluates how "nasty" a pitch is based purely on its physical characteristics — independent of outcomes like balls, strikes, or contact. A Stuff+ of 100 is league average. Above 100 is better than average, below 100 is worse.

This model replicates and extends the Stuff+ methodology using **MLB Statcast data** via `pybaseball`. The core question:

> *Can we predict the run value of a pitch based solely on its physical characteristics at the moment of release?*

**Key features used:**
- Release speed (velocity)
- Induced vertical break (IVB)
- Horizontal break
- Release spin rate
- Release extension
- Release point (x, z coordinates)
- Pitch type

**Target variable:** `delta_run_exp` — the change in run expectancy per pitch (from Statcast)

**Models compared:**
- Random Forest Regressor (baseline)
- HistGradientBoostingRegressor (final model)

**Result:** HistGradientBoostingRegressor outperformed Random Forest on both RMSE and R² — and handles missing values natively, which matters significantly in Statcast data.

---

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install pybaseball scikit-learn pandas numpy matplotlib seaborn plotly --quiet

In [ ]:
# ============================================================
# CELL 2: Imports & Configuration
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from pybaseball import statcast, cache
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

cache.enable()

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')
print('pybaseball cache enabled.')

In [ ]:
# ============================================================
# CELL 3: Data Loading
# Pull Statcast data for the 2024 season
# Uses pybaseball's statcast() function which pulls from Baseball Savant
# ============================================================

print('Loading Statcast data for 2024 season...')
print('Note: First load may take 2-3 minutes. Subsequent loads use cache.')

# Pull full 2024 season in monthly chunks to avoid timeouts
chunks = [
    statcast('2024-03-20', '2024-04-30'),
    statcast('2024-05-01', '2024-06-30'),
    statcast('2024-07-01', '2024-08-31'),
    statcast('2024-09-01', '2024-09-30'),
]

df_raw = pd.concat(chunks, ignore_index=True)
print(f'Raw Statcast data loaded: {len(df_raw):,} pitch events')
print(f'Date range: {df_raw.game_date.min()} to {df_raw.game_date.max()}')

In [ ]:
# ============================================================
# CELL 4: Feature Selection & Cleaning
# ============================================================

# Core Stuff+ features — physical pitch characteristics only
# Deliberately excluding outcome-based features (launch angle, exit velo, etc.)
# This is the key methodological choice: predict quality from release, not result

FEATURES = [
    'release_speed',          # Velocity at release point (mph)
    'release_spin_rate',      # Spin rate (rpm)
    'release_extension',      # Distance from rubber at release (ft)
    'release_pos_x',          # Horizontal release point
    'release_pos_z',          # Vertical release point
    'pfx_x',                  # Horizontal movement vs gravity-only path (in)
    'pfx_z',                  # Induced vertical break (in)
    'plate_x',                # Horizontal location at plate
    'plate_z',                # Vertical location at plate
    'spin_axis',              # Spin axis (degrees)
    'pitch_type',             # Pitch type (categorical)
]

TARGET = 'delta_run_exp'      # Run expectancy change per pitch (Statcast)

# Filter to pitches with a recorded delta_run_exp (plate appearances only)
df = df_raw[FEATURES + [TARGET, 'player_name', 'game_date']].copy()
df = df.dropna(subset=[TARGET])

print(f'Pitches with delta_run_exp: {len(df):,}')
print(f'\nPitch type distribution:')
print(df['pitch_type'].value_counts().head(10))
print(f'\nMissing values per feature:')
print(df[FEATURES].isnull().sum())

In [ ]:
# ============================================================
# CELL 5: Feature Engineering
# ============================================================

# Encode pitch type as numeric
# HistGradientBoostingRegressor handles NaN natively but needs numeric input
le = LabelEncoder()
df['pitch_type_encoded'] = le.fit_transform(df['pitch_type'].fillna('UN'))
print(f'Pitch type encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Add derived features
# Total break magnitude — combines horizontal and vertical movement
df['total_break'] = np.sqrt(df['pfx_x']**2 + df['pfx_z']**2)

# Velocity x IVB interaction — high velocity + high IVB = harder to hit
df['velo_x_ivb'] = df['release_speed'] * df['pfx_z']

# Zone location — distance from center of strike zone
df['zone_distance'] = np.sqrt(df['plate_x']**2 + (df['plate_z'] - 2.5)**2)

FEATURE_COLS = [
    'release_speed', 'release_spin_rate', 'release_extension',
    'release_pos_x', 'release_pos_z', 'pfx_x', 'pfx_z',
    'plate_x', 'plate_z', 'spin_axis', 'pitch_type_encoded',
    'total_break', 'velo_x_ivb', 'zone_distance'
]

print(f'\nFinal feature set ({len(FEATURE_COLS)} features):')
for f in FEATURE_COLS:
    print(f'  - {f}')

In [ ]:
# ============================================================
# CELL 6: Train / Test Split
# ============================================================

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Training set: {len(X_train):,} pitches')
print(f'Test set:     {len(X_test):,} pitches')
print(f'\nTarget variable statistics:')
print(y.describe())

In [ ]:
# ============================================================
# CELL 7: Baseline Model — Random Forest
# ============================================================

print('Training Random Forest baseline...')

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# Random Forest cannot handle NaN — impute with median
X_train_rf = X_train.fillna(X_train.median())
X_test_rf  = X_test.fillna(X_train.median())

rf.fit(X_train_rf, y_train)
y_pred_rf = rf.predict(X_test_rf)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

print(f'\nRandom Forest Results:')
print(f'  RMSE: {rmse_rf:.6f}')
print(f'  MAE:  {mae_rf:.6f}')
print(f'  R²:   {r2_rf:.4f}')

In [ ]:
# ============================================================
# CELL 8: Final Model — HistGradientBoostingRegressor
# ============================================================
#
# WHY HistGradientBoostingRegressor over Random Forest:
#
# 1. Native NaN handling — no imputation required, preserves
#    information in missing values (e.g. missing spin_axis for
#    certain pitch types is informative, not random)
#
# 2. Histogram-based binning — bins continuous features into
#    256 bins before fitting, dramatically faster on large
#    Statcast datasets (700K+ pitches)
#
# 3. Sequential boosting — each tree corrects the residual
#    errors of previous trees, better suited to the structured
#    non-linear relationships in pitch physics
#
# 4. Lower variance — gradient boosting generally overfits
#    less than Random Forest on tabular sports data

print('Training HistGradientBoostingRegressor...')

hgb = HistGradientBoostingRegressor(
    max_iter=300,
    max_depth=6,
    learning_rate=0.05,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=RANDOM_STATE,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)

# No imputation needed — handles NaN natively
hgb.fit(X_train, y_train)
y_pred_hgb = hgb.predict(X_test)

rmse_hgb = np.sqrt(mean_squared_error(y_test, y_pred_hgb))
mae_hgb  = mean_absolute_error(y_test, y_pred_hgb)
r2_hgb   = r2_score(y_test, y_pred_hgb)

print(f'\nHistGradientBoostingRegressor Results:')
print(f'  RMSE: {rmse_hgb:.6f}')
print(f'  MAE:  {mae_hgb:.6f}')
print(f'  R²:   {r2_hgb:.4f}')
print(f'  Iterations used: {hgb.n_iter_}')

In [ ]:
# ============================================================
# CELL 9: Model Comparison
# ============================================================

results = pd.DataFrame({
    'Model': ['Random Forest (Baseline)', 'HistGradientBoosting (Final)'],
    'RMSE': [rmse_rf, rmse_hgb],
    'MAE': [mae_rf, mae_hgb],
    'R²': [r2_rf, r2_hgb],
    'NaN Handling': ['Median imputation', 'Native (no imputation)'],
})

print('=' * 65)
print('MODEL COMPARISON RESULTS')
print('=' * 65)
print(results.to_string(index=False))
print('=' * 65)

rmse_improvement = (rmse_rf - rmse_hgb) / rmse_rf * 100
r2_improvement   = (r2_hgb - r2_rf) / abs(r2_rf) * 100

print(f'\nHistGradientBoosting improvement over Random Forest:')
print(f'  RMSE reduction: {rmse_improvement:.1f}%')
print(f'  R² improvement: {r2_improvement:.1f}%')

# Visualization
fig = go.Figure(data=[
    go.Bar(name='RMSE', x=results['Model'], y=results['RMSE'],
           marker_color=['#636EFA', '#00CC96']),
])
fig.update_layout(
    title='Model Comparison — RMSE (lower is better)',
    template='plotly_dark',
    height=400
)
fig.show()

In [ ]:
# ============================================================
# CELL 10: Feature Importance
# ============================================================

# HistGradientBoosting feature importances
importance_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': hgb.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature Importances (HistGradientBoosting):')
print(importance_df.to_string(index=False))

fig = px.bar(
    importance_df,
    x='Importance', y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='Blues',
    title='Feature Importance — Stuff+ Model (HistGradientBoosting)',
    template='plotly_dark',
    height=500
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# ============================================================
# CELL 11: Generate Stuff+ Scores for 2024 Pitchers
# ============================================================
#
# Stuff+ is scaled so that 100 = league average.
# We compute the model's predicted run value per pitch,
# then normalize to the 100-scale using z-score standardization.
#
# Note: Lower predicted delta_run_exp = better pitch
# (fewer runs expected = harder pitch to hit)
# So we invert the scale: higher Stuff+ = better

# Get predictions for all pitches with sufficient data
df_scored = df.copy()
df_scored['predicted_run_value'] = hgb.predict(X)

# Scale to Stuff+ (100 = league average, invert so higher = better)
mean_rv = df_scored['predicted_run_value'].mean()
std_rv  = df_scored['predicted_run_value'].std()
df_scored['stuff_plus_raw'] = 100 - ((df_scored['predicted_run_value'] - mean_rv) / std_rv * 10)

# Aggregate by pitcher — minimum 200 pitches for qualification
pitcher_stuff = (
    df_scored
    .groupby('player_name')
    .agg(
        stuff_plus=('stuff_plus_raw', 'mean'),
        pitches=('stuff_plus_raw', 'count'),
        avg_velo=('release_speed', 'mean'),
        avg_ivb=('pfx_z', 'mean'),
        avg_spin=('release_spin_rate', 'mean'),
    )
    .reset_index()
)

# Qualify at 200+ pitches
pitcher_stuff_qual = pitcher_stuff[pitcher_stuff['pitches'] >= 200].copy()
pitcher_stuff_qual = pitcher_stuff_qual.sort_values('stuff_plus', ascending=False)

print(f'Qualified pitchers (200+ pitches): {len(pitcher_stuff_qual)}')
print(f'\nTop 20 Pitchers by Stuff+ (2024):')
print(pitcher_stuff_qual.head(20)[['player_name', 'stuff_plus', 'pitches', 'avg_velo', 'avg_ivb']].to_string(index=False))

In [ ]:
# ============================================================
# CELL 12: Stuff+ Leaderboard Visualization
# ============================================================

top30 = pitcher_stuff_qual.head(30).sort_values('stuff_plus')

fig = px.bar(
    top30,
    x='stuff_plus', y='player_name',
    orientation='h',
    color='stuff_plus',
    color_continuous_scale='Blues',
    title='<b>2024 MLB Stuff+ Leaderboard — Top 30 Pitchers</b><br><sup>Model: HistGradientBoostingRegressor on Statcast features | 100 = League Average</sup>',
    labels={'stuff_plus': 'Stuff+', 'player_name': ''},
    template='plotly_dark',
    height=700
)
fig.add_vline(x=100, line_dash='dash', line_color='yellow', annotation_text='League Avg (100)')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

# Stuff+ vs Velocity scatter
fig2 = px.scatter(
    pitcher_stuff_qual,
    x='avg_velo', y='stuff_plus',
    size='pitches', color='avg_ivb',
    hover_name='player_name',
    color_continuous_scale='RdYlGn',
    title='<b>Stuff+ vs Velocity (bubble = pitch count, color = IVB)</b>',
    labels={'avg_velo': 'Avg Velocity (mph)', 'stuff_plus': 'Stuff+', 'avg_ivb': 'Avg IVB'},
    template='plotly_dark',
    height=550
)
fig2.add_hline(y=100, line_dash='dash', line_color='white', annotation_text='League Avg')
fig2.show()

In [ ]:
# ============================================================
# CELL 13: Stuff+ by Pitch Type
# ============================================================

pitch_type_stuff = (
    df_scored
    .groupby('pitch_type')
    .agg(
        avg_stuff_plus=('stuff_plus_raw', 'mean'),
        count=('stuff_plus_raw', 'count'),
        avg_velo=('release_speed', 'mean'),
        avg_spin=('release_spin_rate', 'mean'),
    )
    .reset_index()
    .query('count >= 1000')
    .sort_values('avg_stuff_plus', ascending=False)
)

print('Average Stuff+ by Pitch Type (min 1000 pitches):')
print(pitch_type_stuff.to_string(index=False))

fig = px.bar(
    pitch_type_stuff,
    x='pitch_type', y='avg_stuff_plus',
    color='avg_stuff_plus',
    color_continuous_scale='RdYlGn',
    title='Average Stuff+ by Pitch Type — 2024 MLB',
    template='plotly_dark',
    height=400
)
fig.add_hline(y=100, line_dash='dash', line_color='white', annotation_text='League Avg')
fig.show()

In [ ]:
# ============================================================
# CELL 14: Export Results
# ============================================================

# Export pitcher Stuff+ scores
pitcher_stuff_qual.to_csv('2024_stuff_plus_scores.csv', index=False)
pitch_type_stuff.to_csv('2024_stuff_plus_by_pitch_type.csv', index=False)

print('=== STUFF+ MODEL COMPLETE ===')
print(f'Model: HistGradientBoostingRegressor')
print(f'R²:    {r2_hgb:.4f}')
print(f'RMSE:  {rmse_hgb:.6f}')
print(f'Improvement over Random Forest baseline: {rmse_improvement:.1f}% RMSE reduction')
print(f'Qualified pitchers scored: {len(pitcher_stuff_qual)}')
print()
print('Files exported:')
print('  2024_stuff_plus_scores.csv')
print('  2024_stuff_plus_by_pitch_type.csv')

---

## Methodology Notes

### Why HistGradientBoostingRegressor?

Three reasons drove this model selection over Random Forest:

**1. Native NaN handling.** Statcast data has meaningful missingness — `spin_axis` is not recorded for all pitch types, and `release_extension` is occasionally missing for unusual release points. Random Forest requires imputation, which introduces a decision about what to replace missing values with. HistGradientBoosting learns to route missing values to the optimal branch during training, preserving the signal in missingness itself.

**2. Scale.** A full MLB season contains 700,000+ pitch events. HistGradientBoosting's histogram binning reduces training time by an order of magnitude compared to Random Forest on data of this size, without sacrificing accuracy.

**3. Residual correction.** Gradient boosting sequentially fits trees to correct prior errors, which is well-suited to the structured, non-linear relationships between pitch physics (velocity × break interaction, spin efficiency, etc.) and run value outcomes.

### Limitations

- This model predicts run value from physical characteristics — it is a proxy for Stuff+, not the proprietary Baseball Savant implementation
- Pitcher identity is excluded by design (we want pitch quality, not pitcher reputation)
- Count and game context are excluded — Stuff+ is context-neutral
- 2024 data only — cross-year validation would strengthen the model

### Data Source

All data sourced from MLB Statcast via `pybaseball`. Statcast data is publicly available at [baseballsavant.mlb.com](https://baseballsavant.mlb.com).

---

*Built as SABR Analytics Level IV capstone — Jordan Kurzweil, 2025*